# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

C:\Users\piotr\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\storemagic.py:148: UserWarning: This is now an optional IPython functionality, using autorestore/data_set requires you to install the `pickleshare` library.
  obj = db["autorestore/" + arg]
C:\Users\piotr\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\storemagic.py:148: UserWarning: This is now an optional IPython functionality, using autorestore/labels requires you to install the `pickleshare` library.
  obj = db["autorestore/" + arg]
C:\Users\piotr\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\storemagic.py:148: UserWarning: This is now an optional IPython functionality, using autorestore/test_data_set requires you to install the `pickleshare` library.
  obj = db["autorestore/" + arg]
C:\Users\piotr\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\storemagic.py:148: UserWarning: This is now an optional IPython functionality, using autorestore/test_l

In [2]:
import pickle

with open("data_set.pkl", "rb") as f:
    data_set = pickle.load(f)

with open("labels.pkl", "rb") as f:
    labels = pickle.load(f)

with open("test_data_set.pkl", "rb") as f:
    test_data_set = pickle.load(f)

with open("test_labels.pkl", "rb") as f:
    test_labels = pickle.load(f)

with open("unique_labels.pkl", "rb") as f:
    unique_labels = pickle.load(f)

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [3]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [4]:
def build_classifiers():
    classifiers = [
        LogisticRegression(max_iter=1000),
        KNeighborsClassifier(),
        LinearSVC(),
        DecisionTreeClassifier(random_state=42),
        GaussianNB(),
        QuadraticDiscriminantAnalysis()
    ]

    results = []

    for classifier in classifiers:
        classifier.fit(data_set, labels.reshape((130,)))
        predicted = classifier.predict(test_data_set)
        acc = accuracy_score(test_labels, predicted)
        results.append((acc, classifier))

    results.sort(reverse=True, key=lambda x: x[0])

    return [classifier for acc, classifier in results[:3]]

In [5]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = LogisticRegression(max_iter=1000)
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [6]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

0.95


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [7]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def calculate_accuracy_vector(predicted, labels):
    return (predicted != labels).astype(int)

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [8]:
def set_new_weights(model):
    predicted = model.predict(test_set)
    I = calculate_accuracy_vector(predicted, test_labels)
    return (1 + I) / np.sum(1 + I)

Train the classifier with the code below:

In [9]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00066489 0.00066489 0.00066489 0.00066489 0.00066489 0.00066489
 0.00066489 0.00132979 0.00132979 0.00066489 0.00066489 0.00132979
 0.00066489 0.00132979 0.00066489 0.00066489 0.00132979 0.00132979
 0.00132979 0.00066489 0.00132979 0.00132979 0.00132979 0.00066489
 0.00132979 0.00132979 0.00132979 0.00132979 0.00132979 0.00066489
 0.00132979 0.00132979 0.00066489 0.00132979 0.00066489 0.00132979
 0.00132979 0.00066489 0.00132979 0.00132979 0.00066489 0.00132979
 0.00066489 0.00132979 0.00066489 0.00132979 0.00132979 0.00066489
 0.00066489 0.00066489 0.00066489 0.00132979 0.00132979 0.00066489
 0.00132979 0.00066489 0.00132979 0.00132979 0.00066489 0.00066489
 0.00066489 0.00066489 0.00066489 0.00132979 0.00132979 0.00132979
 0.00132979 0.00066489 0.00132979 0.00066489 0.00132979 0.00132979
 0.00066489 0.00066489 0.00066489 0.00066489 0.00132979 0.00066489
 0.00066489 0.00066489 0.00066489 0.00066489 0.00132979 0.00066489
 0.00066489 0.00066489 0.00066489 0.00066489 0.00066489 0.0013

Set the validation data set:

In [10]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [11]:
def get_prediction(x):
    preds = np.array([model.predict(x) for model in classifiers])
    final = []
    for i in range(preds.shape[1]):
        final.append(np.bincount(preds[:, i]).argmax())
    return np.array(final)

Test it:

In [12]:
prediction = get_prediction(validate_x)[0]

print(prediction)

0
